In [1]:
!pip install pytest torch transformers matplotlib numpy

In [2]:
import torch

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    print(f"Total CUDA devices: {num_devices}")
    for i in range(num_devices):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available.")

Total CUDA devices: 1
Device 0: NVIDIA A100-SXM4-40GB


In [4]:
import torch
import triton

import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [5]:
@triton.jit

def add_kernel(x_ptr,
               y_ptr,
               output_ptr,
               n_elements,
               BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis = 1)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask = mask)
    output = x + y
    tl.store(output_ptr + offsets, output, mask = mask)

In [6]:
def add(x: torch.Tensor, y: torch.Tensor):
    output = torch.empty_like(x)
    assert x.device == DEVICE and y.device == DEVICE and output.device == DEVICE
    n_elements = output.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]))

    add_kernel[grid]